# Pipeline

A two-stage pipeline for interpretable risk scoring of TB drug resistance using a causal feature prior. Causal Mixture Model (CMM) recovers the causal structure linking mutations to MIC; the resulting stability frequencies bias FasterRisk's feature selection toward causally supported predictors.

## Setup

- **Data.** $n = 164$ TB-MDR patients with ~14 prevalent mutations after filtering at prevalence $\in [5\%, 98\%]$, 3 lineage indicators (lineages 1 and 3 merged into the reference), and optionally `type_beyond_MDR` as an exogenous selection-history proxy (sensitivity covariate, off by default).
- **Target.** Continuous $\log_2(\text{dlm\_mic})$ for causal discovery, binarized for downstream classification.

### Justification

- **Log2 MIC.** MIC is measured on a 2-fold dilution series, so $\log_2$ places dilution steps on a uniform scale suitable for linear-Gaussian modeling.
- **Prevalence band $[5\%, 98\%]$.** The lower bound reflects statistical identifiability: mutations carried by fewer than ~8 patients cannot support cluster-conditional coefficient estimation at $k > 1$. The upper bound excludes near-universal mutations (e.g., `rv1979c_A-129G` at 98.8%, `mmpl5_Ile948Val` at 98.2%) that carry no discriminative information.
- **Lineage merging.** Lineages 1 ($n = 3$) and 3 ($n = 4$) are too small for per-lineage effect estimation. Pooling them into the reference category absorbs their effects without producing degenerate estimates.
- **`type_beyond_MDR` (selection-history proxy).** Resistance type is a clinical classification based on prior resistance to first- and second-line drugs (rifampicin, isoniazid, fluoroquinolones), set independently of DLM. It is not a strict confounder of (mutation $\to$ MIC), since type does not cause mutations. Rather, it proxies unobserved selection-history factors (treatment exposure, disease progression) that correlate with both mutation accumulation and MIC. Included as a sensitivity covariate rather than in the default configuration, given collider-bias risk from conditioning on a descendant of other-drug resistance mutations. The 3-level code is collapsed to binary (preXDR + XDR vs MDR, $n = 30$ vs $134$) to avoid fitting structure from $n = 7$ XDR strains.
- **Continuous-then-binary target.** Continuous MIC for the causal stage preserves sub-threshold variation, which carries information about treatment-outcome risk that binary resistance calls discard. Binarization is applied only at the FasterRisk stage, since FasterRisk is a classifier and clinicians act on categorical resistance calls.

## Stage 1: Causal discovery via CMM

Logistic Causal Mixture Model with stability selection.

### Method

CMM extends score-based Bayesian-network learning with per-node latent mixture components. The logistic extension (`FLXMRglm`, binomial link) handles binary mutation predictors directly.

**Structural equation** (logistic mixture, when $Z_i = k$):
$$P(X_j = 1 \mid \mathrm{Pa}_j, Z_i = k) = \sigma\!\left(\beta_{jk}^\top \mathrm{Pa}_j + \beta_{jk}^{(0)}\right)$$

For continuous nodes (MIC), the linear-Gaussian mechanism applies.

### Justification for CMM over alternatives

Standard score-based MB-learning algorithms (SLL, S²TMB, fGES-MB, DMB; see Yu et al., 2020) assume causal sufficiency, i.e., no unobserved confounders. This assumption fails here: patient-level clinical covariates (age, HIV status, treatment history) were not available in time for this work. CMM's per-node mixture components proxy for unobserved confounders by allowing structural equations to vary across $K$ latent clusters, recovering structure under relaxed causal sufficiency. LMB-CSEM is the only standard score-based alternative that admits latent variables, but it uses EM over latent feature subspaces and is computationally prohibitive at $K > 2$.

### Forbidden edges (domain priors)

- $\text{MIC} \to \text{mutation}$ forbidden: mutations precede MIC temporally and mechanistically; the reverse direction is biologically incoherent.
- Edges into `lineage` forbidden: lineage is fixed at infection.
- Edges into `type_beyond_MDR` forbidden: type is a clinical classification set pre-DLM and independently of DLM-relevant mutations.
- Optional: $\text{lineage} \to \text{MIC}$ forbidden when treating lineage strictly as a covariate rather than a putative cause.

Under these constraints MIC is a sink node and the recovered DAG has no descendants of MIC.

### Stability selection

Subsampling without replacement (Meinshausen-Bühlmann, 2010): $B = 100$ runs at 80% subsample fraction. Sparse columns with fewer than $\text{min\_cluster\_count} \times k_{\max}$ positives are dropped per-run before fitting (default $4 \times 6 = 24$ positives). Per-edge frequency $\text{freq}(u \to v)$ is computed with eligibility-weighted denominators (denominator = number of runs in which both endpoints survived the per-run sparse filter).

### Justification of stability-selection choices

- **$B = 100$ runs.** Sufficient for stable edge-frequency estimates at this $p$, consistent with practice in the stability-selection literature (50 to 100 runs).
- **80% subsample fraction.** Meinshausen and Bühlmann (2010) propose subsampling at fraction $\le 1/2$ for finite-sample familywise-error control; we use 80% to preserve sample size for k-component mixture fitting on the smaller cluster, trading the theoretical guarantee for small-sample power at $n = 164$.
- **Per-run column filter at $\text{mcc} \times k_{\max}$ positives.** Below this floor, a $k$-component logistic mixture collapses onto a sparse cluster, producing unidentifiable parameter estimates. The product form follows from the requirement that each of $k$ clusters has at least `mcc` positives to estimate a coefficient.
- **`mcc = 4` at `k_max = 6`.** Empirically retains `rv0678` ($n = 45$, above the $4 \times 6 = 24$ floor) while excluding mutations below the identifiability floor. The choice of $k_{\max} = 6$ is supported by a sweep over $k_{\max} \in \{5, 6, 7\}$: at $k_{\max} = 7$ MIC's modal mixture-component count is still 6 (55 of 100 runs), indicating the natural complexity is captured at $k_{\max} = 6$ rather than artificially capped.
- **Eligibility-weighted denominators.** Standard stability frequency divides by total runs $B$. When per-run feature sets differ (due to the sparse filter), this underestimates frequencies for mutations that survived only a subset of runs. Dividing by the number of runs in which both endpoints were eligible gives an unbiased per-edge estimate.

### Outputs

- Per-edge stability frequency table $\text{freq}(u \to v)$, used directly as the causal prior in Stage 3 (no thresholding applied here).
- Per-node mixture-component count distribution $k_1, \ldots, k_{\max}$.

## Stage 2: Causal feature prior

Under the forbidden-edges design, MIC has no children, so $\text{MB}(\text{MIC}) = \text{parents}(\text{MIC})$ exactly (Aliferis et al., 2010; Yu et al., 2020). The stability frequency $\text{freq}(m \to \text{MIC})$ is interpreted as a subsampling-estimated probability that mutation $m$ lies in MIC's Markov blanket. This continuous frequency is the **causal prior** carried into Stage 3; no hard threshold or set-valued MB estimate is constructed.

### Justification: stability frequency vs alternative causal scores

- **Why direct parents and not ancestors.** The Markov blanket is the information-theoretic optimum for predicting a target variable: features outside it are conditionally independent of the target given the MB. With MIC as a DAG sink, the MB collapses to direct parents, so weighting indirect ancestors would add signal already captured.
- **Why frequency rather than $\beta$ magnitudes.** Stability frequencies aggregate edge presence or absence across subsamples, a coarser quantity than the underlying $\beta_{jk}$ point estimates. In BN learning, DAG structure stabilizes at smaller sample sizes than the parameters it implies; at $n = 164$ ($\sim 131$ per subsample), individual $\beta_{jk}$ values vary substantially across runs while edge-presence stability remains informative. Using frequencies as the prior extracts the part of CMM's output that is most reliable at this scale.

## Stage 3: Causally-penalized FasterRisk

Modify FasterRisk's sparse logistic objective to include a feature-wise causal-prior bonus.

### Objective

Standard FasterRisk:
$$\min_{\lambda \in \mathbb{Z}^p} \; \mathcal{L}_{\text{logistic}}(\lambda) \quad \text{s.t.} \quad \|\lambda\|_0 \leq K, \; |\lambda_m| \leq \lambda_{\max}$$

Causally-penalized version:
$$\min_{\lambda \in \mathbb{Z}^p} \; \mathcal{L}_{\text{logistic}}(\lambda) \; - \; \mu \sum_{m} \text{freq}(m \to \text{MIC}) \cdot \mathbb{1}[\lambda_m \neq 0] \quad \text{s.t. same constraints}$$

Each candidate feature receives a bonus proportional to its subsampling-estimated probability of being in MIC's Markov blanket. At $\mu = 0$ the formulation reduces to vanilla FasterRisk; as $\mu \to \infty$ only features with $\text{freq} > 0$ are selectable (hard pre-selection).

### Justification

- **FasterRisk as the classifier backbone.** Liu et al. (2022) extend RiskSLIM (Ustun and Rudin, 2017) with integer coefficients, bounded magnitudes, and a Rashomon-pool output. These properties match the clinical risk-score paradigm (additive integer points per feature) and yield a model clinicians can read off the coefficients.
- **Why a soft penalty rather than a hard constraint.** A continuous $\mu$ defines a one-parameter family with interpretable limits at both ends (vanilla and hard pre-selection); the limits then serve as natural ablation baselines. A hard constraint discards the uncertainty information in subthreshold stability frequencies (e.g., $\text{freq} = 0.4$ becomes either in or out).
- **Why this functional form.** Linearity and feature-wise separability preserve FasterRisk's beam-search decomposability: per-feature scoring remains computable in isolation, and the per-level greedy step (best children given parents) is unchanged. FasterRisk is a heuristic beam search without formal global-optimality bounds (Liu et al., 2022), so the relevant property to preserve is efficiency, which the penalty leaves intact. Empirical verification on synthetic data is planned (Stage 4).
- **Bayesian interpretation.** With $\mathcal{L}_{\text{logistic}}$ as negative log-likelihood and $\mu \cdot \text{freq}(m \to \text{MIC})$ as the log-prior contribution of including $m$, the modified objective is a MAP estimator under a Bernoulli prior over feature inclusion with parameter $\sigma(\mu \cdot \text{freq})$.

### Implementation

Two localized modifications in [`external/fasterrisk/src/fasterrisk/`](../../external/fasterrisk/src/fasterrisk/):

- [`sparseBeamSearch.py`](../../external/fasterrisk/src/fasterrisk/sparseBeamSearch.py): add the per-feature bonus to (i) the gradient-magnitude ranking used to select candidate features for expansion, and (ii) the child-loss computation used to rank candidate supports.
- [`sparseDiversePool.py`](../../external/fasterrisk/src/fasterrisk/sparseDiversePool.py): mirror the same change in the Rashomon-set generation step.

The integer rounding step is unaffected, since rounding adjusts magnitudes rather than support.

### Outputs

Rashomon pool $\Lambda^* = \{\lambda^{(1)}, \ldots, \lambda^{(R)}\}$ of near-optimal causally-regularized sparse integer solutions.

## Stage 4: Evaluation

### Hyperparameter sweep

$\mu \in \{0, 0.01, 0.1, 1, 10, \infty\}$, evaluated on cross-validated AUC, Brier score, sparsity, and selected feature set. Logarithmic spacing covers the regimes from no causal prior ($\mu = 0$), through mild and substantive bias, to causally-dominated selection ($\mu \to \infty$).

### Binarization threshold sensitivity

The continuous-to-binary target conversion at Stage 3 is itself a modeling choice. Three candidates evaluated on AUC, Brier, calibration, and biological plausibility of selected features:

- **Median split.** Balanced classes (~50/50). Maximizes statistical power; clinically arbitrary.
- **Top quartile MIC.** Captures elevated MIC without invoking the clinical cut-off; ~25% positive class.
- **`interp_dlm016`** (agar-based binary). Clinically meaningful but disagrees with MIC for ~32 strains.

Reporting on multiple thresholds checks that conclusions are not artifacts of one binarization choice.

### Synthetic-data verification

Before deploying on TB data, validate on simulated data with planted causal signal: a known sparse linear-Gaussian DAG with a designated target, mutations sampled with controlled prevalence. Verify that increasing $\mu$ progressively shifts FasterRisk's feature selection toward the planted causal parents (recovery rate as a function of $\mu$). This is the main check against implementation bugs in the modified beam search.

### Ablations

| Ablation | Isolates |
|---|---|
| Vanilla FasterRisk ($\mu = 0$) | The contribution of the causal penalty |
| Hard pre-selection ($\mu = \infty$) | The value of the continuous-vs-thresholded interpolation |
| Uniform prior ($\text{freq}(m) = $ const) | Whether the *structure* of the prior carries information beyond regularization strength |
| With/without lineage covariate | Sensitivity to population-structure confounding |
| With/without `type_beyond_MDR` | Sensitivity to selection-history adjustment (proxy for unobserved treatment exposure / disease progression; off in the default configuration) |

### Justification of evaluation choices

- **Cross-validation over a held-out test set.** $n = 164$ is too small to sacrifice a substantial held-out fraction. 5-fold CV uses available data for both training and evaluation, consistent with small-$n$ medical-ML practice.
- **AUC + Brier.** AUC measures ranking quality (robust to class imbalance), Brier measures calibration. Reporting both avoids the case of a well-ranking but miscalibrated risk score.
- **Uniform-prior ablation.** Without it, a positive effect of the causal penalty could be attributed to regularization in general rather than to the structural information in the stability frequencies.

### Clinical sanity check

Present the top-ranked Rashomon solutions to clinical collaborators for biological plausibility, checking against known resistance genes (`rv0678`, `pepQ`, `atpE`, `fbiA`-D, `fgd1`, `ddn`).

## Scope and limitations

- **Rare mutations excluded.** With $n = 164$, mutations below ~5% prevalence cannot support statistical inference under any method. Characterizing rare mutations requires substantially larger cohorts and is out of scope.
- **Patient covariates unavailable.** Clinical covariates (age, HIV status, treatment history) not yet harmonized at the time of this work. CMM's latent mixture components partially compensate by absorbing residual heterogeneity. Future work integrating these covariates will allow direct adjustment.
- **DLM-only.** Restricted to delamanid resistance. Extension to BDQ, CFZ, LNZ, PTM is a pipeline rerun once the corresponding MIC data is consolidated.
- **Faithfulness assumption.** As with all causal discovery, recovered structure assumes the joint distribution's conditional independencies match those implied by the DAG.

### Anticipated risks and defenses

- **Sample-size bottleneck at $n = 164$ ($\sim 131$ per subsample).** Fitting $k$-component mixtures at this scale produces individual $\beta_{jk}$ point estimates that are highly variable across runs. *Defense:* the pipeline never propagates individual $\beta_{jk}$ past Stage 1. Stability frequencies aggregate edge presence or absence, which stabilizes at smaller sample sizes than the parameters underlying the DAG. Using frequencies as the prior extracts the part of CMM's output that is most reliable at this scale.

- **Continuous-to-binary target mismatch.** Stage 1 builds the causal prior on $\log_2(\text{MIC})$ while Stage 3 trains on a binarized resistance call. A mutation with strong causal effect on MIC (e.g., shifting it from 0.125 to 0.5) may receive coefficient 0 from FasterRisk if both endpoints lie on the same side of the threshold. *Defense (architectural):* this is the intended behavior of the two-stage design, not a failure mode. Stage 1 recovers full causal structure; Stage 3 keeps only the subset clinically actionable under a binary decision. *Defense (empirical):* threshold selection is itself a modeling choice and is evaluated in Stage 4 (binarization-threshold sensitivity analysis). If conclusions hold across median split, top-quartile, and `interp_dlm016`, the pipeline is robust to threshold choice. If they do not, the threshold becomes a primary modeling decision.

- **Decomposability preservation in modified FasterRisk.** Injecting a custom penalty into FasterRisk's beam search must not break the per-feature scoring structure on which the search depends for efficiency. *Defense:* the penalty $\mu \sum_m \text{freq}(m \to \text{MIC}) \cdot \mathbb{1}[\lambda_m \neq 0]$ is linearly separable across features, so the marginal cost of any candidate feature can still be computed without recomputing global support quantities. Per-level greedy selection is unchanged. Verification is empirical: synthetic-data recovery experiments confirm that the penalty steers selection toward planted causal features (Stage 4).

- **Collider bias from `type_beyond_MDR`.** Type is a descendant of resistance mutations targeting first- and second-line drugs. Conditioning on it can open a spurious backdoor path from those resistance mutations into the model. *Defense:* type is run as a sensitivity covariate, not in the default configuration. If top-ranked MIC parents are stable with and without type, the recovery is robust to this adjustment. If lineage_2's frequency drops when type is added, that is informative: lineage_2's signal is partly mediated by treatment history rather than directly causal.

## Methodological contributions

1. **Logistic extension of CMM for binary observed variables.** FLXMRglm binomial link integrated into the mixture-EM structure-learning loop, with sparse-column handling and stochastic-EM jittering for numerical robustness on small samples (Diebolt and Celeux, 1993).
2. **Stability-frequency causal prior for sparse interpretable risk scoring.** A one-parameter family of risk-score classifiers regularized toward features with structural causal support, interpolating between vanilla FasterRisk ($\mu = 0$) and hard Markov-blanket pre-selection ($\mu = \infty$).
3. **Latent-variable causal discovery framework for partial covariate availability.** CMM's mixture components proxy for unobserved patient-level confounders, allowing causal structure recovery under relaxed causal sufficiency, a regime where standard score-based MB learning (SLL, fGES-MB) cannot operate without explicit latent-variable estimation.

## Potential extensions (out of scope)

- **Co-parentship interactions.** For mutation pairs $(m, m')$ sharing a stable common child in the DAG, generate interaction features $x_{mm'} = x_m \cdot x_{m'}$. Biologically motivated for shared-mechanism resistance.
- **Multi-outcome causal prior.** Use $\max$ or $\min$ over $\{\text{freq}(m \to \text{MIC}), \text{freq}(m \to \text{prop\_mutants})\}$ for union or intersection priors. Blocked currently by data-quality issues with `prop_mutants_dlm`.
- **Hard causal constraints in FasterRisk.** Forbid coefficient pairs $(\lambda_m, \lambda_{m'})$ that violate d-separation in $G^*$. More aggressive than the soft penalty; deferred until the soft version is validated.
- **Path-discounted ancestor prior.** Replace direct-parent stability with ancestor frequencies discounted by path length. Tests whether indirect causes carry signal beyond Markov blanket sufficiency.